# Structure Inference Agent (SIA) - Technical Specification

This notebook contains the technical specification and workflow for the Structure Inference Agent (SIA), designed to process messy Excel workbooks.

## User Prompt

The following prompt was used to generate this specification:

```text

***

You are an expert AI architect and agent designer. Your task is to design a *technical specification and workflow* for a **Structure Inference Agent (SIA)** that takes messy, semi-structured Excel workbooks and deterministically produces a normalized relational schema plus a flattened data payload, suitable for downstream ETL.

Design this as if you are creating a production-grade agent that will be evaluated primarily on **accuracy, reliability, and error handling**, not on UI.

## 1. Objective and Scope

- The SIA must analyze semi-structured tabular data sources (complex Excel workbooks with varying layouts, multiple headers, merged cells, disjoint blocks, multi-sheet workbooks) and infer a **normalized relational schema** and a **combined normalized dataset**.
- The system should move from manual ETL mapping toward an AI-driven approach that learns structural patterns from:
  - Declarative, semantic instructions (annotation guidelines).
  - Labeled ground-truth samples (input → expected metadata).

Clearly restate and refine this objective, emphasizing:
- Deterministic, auditable transformations.
- Separation between **reasoning/structure inference** and **deterministic data extraction**.
- Focus on handling real-world “messy Excel” scenarios.

## 2. Training / Configuration Inputs

Design how the agent is configured and “trained” using two input classes:

### A. Declarative Instructions (Semantic Guidelines)

Define a schema for **semantic** annotation guidelines, not brittle row/column indices. Cover at minimum:

- **Data Block Identification**
  - Criteria for identifying contiguous ranges of relevant data vs. metadata/headers/footers/empty space.
  - Examples of semantic rules like:
    - “Ignore report title regions that contain long text blocks above the main numeric table.”
    - “Treat dense rectangular regions with consistent data types as candidate data blocks.”

- **Attribute Classification**
  - Distinguish:
    - **Dimensions**: qualitative attributes used for grouping/filtering (Region, Product ID).
    - **Metrics**: quantitative attributes typically aggregated (Sales Amount, Unit Count).
  - Rules for mapping header strings + column data types to these roles.

- **Temporal Recognition**
  - Expected date/time formats (e.g., `YYYY-MM-DD`, `MM/DD/YY`, `Mon-YY`).
  - Guidance on identifying “temporal anchors” (record date columns, period columns, etc.).

- **Multi-Sheet Consolidation Strategy**
  - Semantic rules for merging across sheets:
    - Example: “Tabs named Q1, Q2, Q3, Q4 are temporal partitions of the same dataset; align schemas and vertically append.”
    - Generalize to sheets with similar layout and semantics.

Your output should propose:
- A structured configuration spec (e.g., a JSON/YAML shape) that captures these semantic guidelines.
- Concrete examples of good vs. bad instructions (semantic vs. hard-coded row/column instructions).

### B. Ground-Truth Samples (Labeled Datasets)

Design how **labeled examples** are represented and used:

- Each example is a pair:
  - **Raw input**: e.g., specific workbook + named sheet + approximate region.
  - **Expected metadata labels**, such as:
    - `Header_Row`
    - `Data_Range`
    - `Date_Column` with format mask
    - `Metric_Columns` with names/roles
- Show a sample representation for:
  - Input reference: `Sheet1!A1:F20`.
  - Output labels:
    - `Header_Row: 1`
    - `Data_Range: A2:F20`
    - `Date_Column: A (Format: YYYY-MM-DD)`
    - `Metric_Column: E ("Sales_Amount")`

Address the **annotation bottleneck**:

- Propose a strategy that **does not rely on large fully supervised datasets alone**, but instead uses:
  - A **few-shot + RAG** approach:
    - A vector DB (or equivalent) of prior successful Excel → schema mappings.
    - Retrieval of top-k similar historical examples for a new file.
    - Construction of prompts such as: “Here are 3 similar examples (input snippet → schema). Here is the new grid snippet. Infer blocks and attributes following these patterns.”
- Explain how corrected outputs from humans become new labeled samples to improve future performance.

## 3. Core Agent Architecture and Modules

Design the SIA as a **modular, tool-calling agent**. For a new, unseen workbook, it should execute a chain of internal “tools” / modules with clear interfaces and error handling.

Describe each module in detail (inputs, outputs, main logic, failure modes):

### 3.1 Visual Pre-processing (Critical for Messy Excel)

Introduce an explicit **visual flattening / normalization** step before any reasoning:

- Load Excel with a robust library (e.g., openpyxl/pandas).
- Handle:
  - **Merged cells**: unmerge and propagate header values across the merged range.
  - **Hidden rows/columns**: decide whether to ignore or surface them according to instructions.
  - **Cell formatting metadata**: optionally capture text, style (bold, color), which may have semantic value.
- Output: a **dense 2D text grid** + metadata (per-cell type, style, original coordinates) that subsequent modules operate on.

Define:
- Data structures for the visual grid.
- How errors (corrupt workbook, unsupported formats) are surfaced and handled.

### 3.2 locate_data_blocks(file_handle or visual_grid)

Refine the original `locate_data_blocks` into a robust module that:

- Scans the 2D grid for **candidate table regions** based on:
  - Cell density.
  - Type consistency by row/column.
  - Separation from large-text “decorative” zones.
- Uses both:
  - **Semantic guidelines** (from instructions).
  - **Few-shot / retrieved examples** for similar layouts.
- Produces:
  - A set of candidate data blocks with coordinates and confidence scores.
  - Explanations / features used (e.g., “high numeric density”, “header-like row followed by homogeneous numeric rows”).

Include:
- How uncertainty is represented.
- How to handle multiple candidate blocks (e.g., multiple tables in a single sheet).

### 3.3 classify_attributes(block_headers)

Define this module to:

- Analyze header strings + underlying column samples to classify each column as:
  - Dimension, Metric, Temporal, or Other.
- Use:
  - Semantic patterns in header text.
  - Data type / value distributions in the column.
  - Retrieved examples illustrating similar column semantics.
- Output:
  - For each column: name, semantic role, data type (string, integer, float, date, etc.), confidence.

Explain:
- Error behavior when headers are missing or ambiguous.
- How ambiguous columns are flagged for human review.

### 3.4 normalize_temporal_data(date_columns)

Define logic to:

- Detect possible date/time fields.
- Infer their format masks (e.g., `DD/MM/YYYY`, `Mon-YY`).
- Normalize into a canonical representation, e.g., ISO-8601.
- Return:
  - Parsed values.
  - Original raw values.
  - Confidence and any rows that failed parsing.

Specify:
- Use of deterministic date parsing libraries.
- How to avoid relying solely on the LLM for bulk parsing (LLM used for reasoning about formats, not row-wise transformation).

### 3.5 merge_sheet_data(workbook_metadata)

Define the consolidation logic:

- Detect structurally similar sheets based on:
  - Sheet names (e.g., “Q1”, “Q2”, “2023_Jan”).
  - Layout and header similarity.
- Align schemas and vertically append rows into a single normalized table when appropriate.
- Handle:
  - Slight column ordering differences.
  - Missing columns filled with nulls or defaults.
- Output:
  - A unified data structure (e.g., pandas DataFrame) conforming to the inferred schema.

Explain:
- How conflicts and schema mismatches are detected.
- How low-confidence merges get routed to human-in-the-loop.

## 4. Instruction vs. Inference: Design Principles

Explicitly articulate how to separate:

- **Hard rules / configuration** (simple deterministic logic) from
- **Inference / reasoning** (where the LLM or learned model is valuable).

Provide examples:

- Replace prescriptive rules like “Ignore first 3 rows” with semantic heuristics like:
  - “Report headers are long text segments above the first dense numeric table.”
- Show how semantic guidelines are incorporated in the system prompt / control flow, while final decisions still depend on the current sheet content.

## 5. Uncertainty, Confidence, and Human-in-the-Loop (HITL)

Design a **confidence and exception-handling layer**:

- Each major module (locate_data_blocks, classify_attributes, normalize_temporal_data, merge_sheet_data) should output:
  - A **confidence score**.
  - Structured rationales / signals used for the decision.
- Define:
  - A global policy for routing low-confidence cases (e.g., < 0.9) into a **HITL review queue**.
- Describe the **HITL interface/logical contract**:
  - What the human sees (grid preview, proposed schema, detected blocks).
  - What corrections they can make.
  - How corrections are stored back as new labeled examples for future retrieval.

## 6. Final Output Contracts

Specify, in detail, the final two-part output:

### 6.1 Inferred Schema Definition (Metadata)

- A formal JSON Schema–like object defining:
  - `schema_name`
  - `fields`: for each field:
    - `name`
    - `type` (string, integer, float, date, etc.)
    - `format` (for dates)
    - `role` (dimension/metric/temporal)
    - Optional: `nullable`, `description`, `example_values`, `source_coordinates`.
- Use the “Sales_Consolidated” example as a pattern, but generalize it.

### 6.2 Combined Normalized Data (Payload)

- A single flattened dataset (e.g., pandas DataFrame/CSV/Parquet) that:
  - Strictly adheres to the inferred schema.
  - Contains all merged rows from relevant sheets and blocks.
- Emphasize that:
  - The **LLM is not used to transform bulk rows**.
  - Deterministic code (e.g., pandas) applies the schema’s coordinates, mappings, and parsed formats to the full dataset.

Define:
- Error conditions (e.g., rows that cannot be coerced to schema types).
- How to surface data quality issues (e.g., bad dates) with logs and metrics.

## 7. Reliability, Error Handling, and Observability

Design for robustness:

- **Error Types & Handling**
  - Corrupt or unsupported files.
  - Inconsistent merged cells or formula errors.
  - Ambiguous header detection.
  - Date parsing failures.
- For each, define:
  - Detection method.
  - User-facing error messages.
  - Fallback strategies (e.g., partial schema, HITL escalation).

- **Logging and Tracing**
  - Log each module’s input, output, confidence, and any exceptions.
  - Provide an execution trace that can be shown in a debugging UI:
    - Step-by-step decisions.
    - Retrieved examples.
    - Rationale summaries for key inferences.

## 8. Few-Shot + RAG Design for Structure Inference

Describe in detail how to combine:

- A **powerful LLM** (e.g., Claude 3.5/Opus-level) as the central reasoning component.
- A **retrieval layer**:
  - Index historical Excel snippets + schemas as text/structured representations.
  - On each new file, retrieve the most similar k cases.
- A **prompting strategy** that:
  - Injects semantic guidelines.
  - Adds retrieved examples (before/after) as demonstrations.
  - Presents the current 2D grid snapshot.
  - Asks the LLM to output:
    - Proposed schema JSON.
    - Mapping from original grid coordinates to schema fields.
    - Confidence and rationale.

Explain:
- How this hybrid approach reduces the need for large, fully supervised training sets while still capturing domain-specific structural patterns.

***

```


# Structure Inference Agent (SIA) - Technical Specification

## 1. Objective and Scope

The **Structure Inference Agent (SIA)** is a production-grade system designed to transform messy, semi-structured Excel workbooks into normalized, relational datasets. The core philosophy is **"Probabilistic Inference, Deterministic Execution"**.

**Refined Objective:**
To autonomously analyze complex Excel files (merged cells, multi-headers, disjoint blocks) and infer a strict relational schema using semantic reasoning and historical examples. Once the schema and structural mappings are inferred and validated (by high confidence or human review), the actual data extraction is performed by **deterministic, audit-logged code**, ensuring 100% reproducibility and traceability.

**Key Scope Constraints:**
*   **Input:** `.xlsx`, `.xls`, `.csv` files with arbitrary layouts.
*   **Output:** 
    1.  **Schema Definition:** JSON object describing tables, fields, types, and constraints.
    2.  **Normalized Data:** A flat dataframe (e.g., Parquet/Pandas) adhering strictly to the schema.
*   **Non-Goal:** The LLM does *not* process every row of data. It only defines the *rules* (schema & mapping) for the deterministic processor.

---

## 2. Training / Configuration Inputs

### A. Declarative Instructions (Semantic Guidelines)

We define a `SemanticConfig` object (YAML/JSON) that guides the agent without hard-coding coordinates.

**Configuration Schema Example:**

```yaml
semantic_guidelines:
  data_block_identification:
    ignore_regions:
      - "Header areas with > 80% empty cells"
      - "Footer notes starting with 'Source:' or 'Confidential'"
    candidate_criteria:
      - "Rectangular regions with > 5 rows and > 2 columns"
      - "Regions where column data types are consistent (> 90% match)"
  
  attribute_classification:
    dimensions:
      keywords: ["Region", "Category", "SKU", "Name"]
      max_cardinality_ratio: 0.2  # Heuristic: Dimensions usually have repeated values
    metrics:
      keywords: ["Sales", "Revenue", "Qty", "Amount", "Total"]
      data_types: ["int", "float", "currency"]
  
  temporal_recognition:
    anchor_keywords: ["Date", "Period", "Year", "Month"]
    expected_formats: ["%Y-%m-%d", "%d/%m/%Y", "%b-%y"]
  
  multi_sheet_strategy:
    consolidation_mode: "auto_append"
    partition_key_inference: true # e.g., if sheet is named "Q1", add column "SheetName"='Q1'
```

**Good vs. Bad Instructions:**
*   **Bad (Brittle):** "Data always starts at Row 5."
*   **Good (Semantic):** "Data begins after the first row containing 'Product ID' or 'Date'."

### B. Ground-Truth Samples (Labeled Datasets)

Labeled examples are stored in a **Vector Store** (e.g., ChromaDB, Pinecone) to enable Few-Shot RAG.

**Example Representation:**

```json
{
  "example_id": "fin_report_001",
  "input_context": {
    "sheet_name": "Q1_2023",
    "grid_snippet": "[[null, 'Report', null], ['Date', 'Item', 'Cost'], ['2023-01-01', 'A', 100]]",
    "visual_features": "Header row is bold. 'Report' is merged A1:C1."
  },
  "expected_metadata": {
    "header_row_index": 1, 
    "data_range": "A2:C100",
    "columns": [
      {"name": "Date", "role": "temporal", "format": "%Y-%m-%d"},
      {"name": "Item", "role": "dimension"},
      {"name": "Cost", "role": "metric"}
    ]
  }
}
```

**Annotation Strategy:**
1.  **Cold Start:** Manually annotate 20-50 diverse examples.
2.  **Active Learning:** When the agent's confidence is low (< 0.8), the case is sent to HITL. The human's correction is saved as a new "Gold Standard" example and embedded into the vector store, immediately improving performance on similar future files.

---

## 3. Core Agent Architecture and Modules

The agent operates as a **Directed Acyclic Graph (DAG)** of tools.

### 3.1 Module: Visual Pre-processing (`visual_normalizer`)

**Goal:** Convert specific Excel binary formats into a standardized "Visual Grid" for the LLM.

*   **Logic:**
    1.  Load workbook using `openpyxl`.
    2.  **Unmerge Cells:** Duplicate the value of the top-left cell to all cells in the merged range. Record `is_merged=True` in metadata.
    3.  **Style Extraction:** Extract `is_bold`, `bg_color`, `indent_level`. These are strong semantic signals (e.g., bold often equals header).
    4.  **Grid Construction:** Create a 2D array of `Cell` objects.
*   **Output:** `VisualGrid` object.
    ```python
    class Cell:
        value: Any
        row: int
        col: int
        style: Dict[str, Any] # {bold: True, color: 'FF0000'}
        original_type: str
    ```

### 3.2 Module: `locate_data_blocks`

**Goal:** Identify the bounding boxes of data tables.

*   **Input:** `VisualGrid`, `SemanticConfig`, `RetrievedExamples`
*   **Logic:**
    1.  **Heuristic Scan:** Identify islands of non-empty cells.
    2.  **LLM Inference:** Pass a simplified text representation of the grid (e.g., first 20 rows) + retrieved examples to the LLM.
    3.  **Prompt:** "Given these examples of headers and data blocks, identify the start/end coordinates of the main table in this grid."
*   **Output:** List of `DataBlock` candidates.
    ```json
    [
      {
        "range": "A5:K100",
        "header_row": 4,
        "confidence": 0.95,
        "type": "main_table"
      }
    ]
    ```

### 3.3 Module: `classify_attributes`

**Goal:** Assign semantic meaning to columns.

*   **Input:** `DataBlock` headers + sample data rows.
*   **Logic:**
    1.  Extract header text and 5 sample values for each column.
    2.  **LLM Inference:** "Classify column 'Inv Date' with samples ['2023-01-01', ...] as Dimension, Metric, or Temporal."
    3.  **Validation:** Check if inferred type matches sample data (e.g., if LLM says "Metric" but data is "ABC", flag error).
*   **Output:** `ColumnSchema` list.

### 3.4 Module: `normalize_temporal_data`

**Goal:** Deterministic date parsing.

*   **Input:** Columns identified as `Temporal`.
*   **Logic:**
    1.  **Pattern Discovery:** Use LLM to suggest format string (e.g., "It looks like DD-Mon-YY").
    2.  **Deterministic Verify:** Try parsing the column using Python's `datetime.strptime` with the suggested format.
    3.  **Fallback:** If > 5% failure rate, try `dateutil.parser` or pandas `to_datetime(errors='coerce')`.
*   **Output:** Standardized ISO-8601 dates. Rows that fail parsing are flagged in an `error_log`.

### 3.5 Module: `merge_sheet_data`

**Goal:** Consolidate multi-sheet workbooks.

*   **Logic:**
    1.  **Clustering:** Group sheets by schema similarity (Jaccard similarity of header sets).
    2.  **Alignment:** For sheets in the same cluster, map columns to the canonical schema.
    3.  **Vertical Concat:** Append data. Add `_source_sheet` column for lineage.
*   **Conflict Resolution:** If Sheet A has column "Tax" and Sheet B doesn't, fill with `null` (if nullable) or flag for HITL if critical.

---

## 4. Instruction vs. Inference: Design Principles

We strictly separate **Policy** (Instructions) from **Mechanism** (Inference).

*   **Mechanism (The LLM):** "Here is a grid. Based on the policy 'Ignore legal disclaimers', where does the data start?"
*   **Policy (The Config):** The definition of what a "legal disclaimer" looks like is injected via the System Prompt or RAG.

**Example:**
*   *Instruction:* "Fiscal Year headers often appear in Row 1, separate from the main table."
*   *Inference:* LLM sees "FY2023" in A1 and correctly tags it as metadata, not a column header.

---

## 5. Uncertainty, Confidence, and HITL

**Confidence Scoring:**
*   **Block Confidence:** `(Density Score + LLM LogProb) / 2`
*   **Schema Confidence:** Average confidence of column classifications.

**Routing Logic:**
*   `Confidence > 0.9`: **Auto-Approve**. Proceed to extraction.
*   `0.6 < Confidence <= 0.9`: **Flag for Review**. Process but mark as "Provisional".
*   `Confidence <= 0.6`: **Stop & Ask**. Send to HITL queue.

**HITL Interface:**
*   **Visual:** User sees the Excel grid with an overlay of the inferred bounding box.
*   **Action:** User can drag-and-drop to resize the box or rename columns.
*   **Feedback Loop:** User actions are recorded as: `{"input": grid_state, "output": user_corrected_schema}` and added to the training set.

---

## 6. Final Output Contracts

### 6.1 Inferred Schema Definition (Metadata)
```json
{
  "schema_name": "Sales_Consolidated",
  "version": "1.0",
  "fields": [
    {
      "name": "transaction_date",
      "type": "date",
      "format": "%Y-%m-%d",
      "role": "temporal",
      "source_coordinates": {"sheet": "*", "col_header": "Date"}
    },
    {
      "name": "amount",
      "type": "float",
      "role": "metric",
      "nullable": false
    }
  ]
}
```

### 6.2 Combined Normalized Data (Payload)
A **Parquet** file containing the clean data.
*   **Guarantees:**
    *   All dates are ISO-8601.
    *   Numeric columns are actual numbers (no "$1,000" strings).
    *   Schema violations (e.g., text in a float column) result in `null` + entry in `error_log`.

---

## 7. Reliability, Error Handling, and Observability

**Error Handling Matrix:**

| Error Type | Detection | Action |
| :--- | :--- | :--- |
| **Corrupt File** | `openpyxl` exception | Reject file, alert user. |
| **Ambiguous Header** | Multiple rows look like headers | Low confidence flag -> HITL. |
| **Date Parse Fail** | `strptime` raises ValueError | Nullify value, add to `row_errors` list. |

**Observability:**
*   **Trace ID:** Every file processing run has a UUID.
*   **Step Logs:**
    *   `[INFO] Visual Pre-processing: Found 3 merged ranges.`
    *   `[INFO] Block Location: Found candidate A5:F20 (Conf: 0.98).`
    *   `[WARN] Date Normalization: 5 rows failed parsing in col 'Date'.`

---

## 8. Few-Shot + RAG Design

**Architecture:**
1.  **Query:** New Workbook Snippet (Text representation of top 20 rows).
2.  **Retrieval:** Embed snippet using `text-embedding-3-small`. Query Vector DB for top-3 similar past snippets.
3.  **Prompt Construction:**
    ```text
    System: You are a Data Engineer. Extract schema info.
    
    Example 1 (Retrieved):
    Input: [Grid Snippet A]
    Output: {header: 3, columns: [...]}
    
    Example 2 (Retrieved):
    Input: [Grid Snippet B]
    Output: {header: 1, columns: [...]}
    
    Current Task:
    Input: [Current Grid Snippet]
    Output: ?
    ```
4.  **Reasoning:** The LLM uses the examples to understand *context-specific* patterns (e.g., "In this company's reports, the date is always in the top right").

This approach allows the agent to "learn" new formats instantly without re-training a model, simply by adding a corrected example to the database.
